In [1]:
!pip install -q nltk indic-nlp-library


In [1]:
import nltk
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger
from indicnlp.tokenize.indic_tokenize import trivial_tokenize

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

In [2]:
def parse_conllu(file_path):
    sentences = []
    sentence = []

    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()

            # Sentence boundary
            if not line or line.startswith('#'):
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue

            parts = line.split('\t')

            # Ensure valid CoNLL-U format
            if len(parts) == 10:
                word = parts[1]
                pos_tag = parts[3]

                # Skip special tokens like 1-2 (multiword tokens)
                if '-' in parts[0] or '.' in parts[0]:
                    continue

                sentence.append((word, pos_tag))

    if sentence:
        sentences.append(sentence)

    return sentences

In [3]:
train_file = "hi_hdtb-ud-train.conllu"
test_file = "hi_hdtb-ud-test.conllu"

print(" Loading dataset...")

train_sents = parse_conllu(train_file)
test_sents = parse_conllu(test_file)

print(f"Train sentences: {len(train_sents)}")
print(f"Test sentences: {len(test_sents)}")


 Loading dataset...
Train sentences: 13306
Test sentences: 1684


In [4]:
print("\n Training taggers...")

default_tagger = DefaultTagger('X')  # better fallback than NOUN

unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
bigram_tagger = BigramTagger(train_sents, backoff=unigram_tagger)
trigram_tagger = TrigramTagger(train_sents, backoff=bigram_tagger)

print(" Training complete!")




 Training taggers...
 Training complete!


In [5]:
print("\n Evaluating model...")

print(f"Unigram Accuracy : {unigram_tagger.accuracy(test_sents):.2%}")
print(f"Bigram Accuracy  : {bigram_tagger.accuracy(test_sents):.2%}")
print(f"Trigram Accuracy : {trigram_tagger.accuracy(test_sents):.2%}")



 Evaluating model...
Unigram Accuracy : 89.82%
Bigram Accuracy  : 90.69%
Trigram Accuracy : 90.73%


In [6]:
def tag_sentence(sentence):
    tokens = trivial_tokenize(sentence, lang='hi')
    return trigram_tagger.tag(tokens)

In [7]:
print("\n Sample Predictions:\n")

samples = [
    "दो आदमी आए।",
    "राम स्कूल जाता है।",
    "भारत एक महान देश है।"
]

for sent in samples:
    tagged = tag_sentence(sent)
    formatted = " ".join([f"{w}/{t}" for w, t in tagged])
    print(f"Input : {sent}")
    print(f"Output: {formatted}\n")


 Sample Predictions:

Input : दो आदमी आए।
Output: दो/NUM आदमी/NOUN आए/VERB ।/PUNCT

Input : राम स्कूल जाता है।
Output: राम/PROPN स्कूल/PROPN जाता/AUX है/AUX ।/PUNCT

Input : भारत एक महान देश है।
Output: भारत/PROPN एक/NUM महान/ADJ देश/NOUN है/AUX ।/PUNCT



In [8]:
def interactive_mode():
    print("\n Enter Hindi sentence (type 'exit' to quit)\n")

    while True:
        try:
            user_input = input(">>> ")

            if user_input.lower() == 'exit':
                print(" Exiting...")
                break

            tagged = tag_sentence(user_input)
            print("Tagged:", " ".join([f"{w}/{t}" for w, t in tagged]))

        except KeyboardInterrupt:
            print("\n Interrupted. Exiting...")
            break



In [9]:
interactive_mode()


 Enter Hindi sentence (type 'exit' to quit)



>>>  राम स्कूल जाता है और सीता घर पर रहती है।


Tagged: राम/PROPN स्कूल/PROPN जाता/AUX है/AUX और/CCONJ सीता/PROPN घर/NOUN पर/ADP रहती/VERB है/AUX ।/PUNCT


>>>  exit


 Exiting...


    "भारत एक महान देश है।",
    "मुंबई भारत का एक बड़ा शहर है।",
    "गांधीजी एक महान नेता थे।"
    "राम स्कूल जाता है और सीता घर पर रहती है।",
    "जब बारिश होती है तब लोग घर में रहते हैं।",
    "अगर तुम पढ़ोगे तो पास हो जाओगे।"